In [1]:
import pyomo.environ as pyo
from pyomo.environ import *
from pyomo.environ import SolverFactory

In [3]:
produtos = ['portas','janelas']
requisitos = ['corte','jateamento','acabamento']

valores_req = {
    requisitos[0]:40*60,
    requisitos[1]:40*60,
    requisitos[2]:60*60,
}

informacoes = {
    produtos[0]:{
        requisitos[0]:1*60,
        requisitos[1]:30,
        requisitos[2]:30,
        'lucro':500
    },
    produtos[1]:{
        requisitos[0]:30,
        requisitos[1]:45,
        requisitos[2]:1*60,
        'lucro':400
    }
    
}

In [9]:
model = pyo.ConcreteModel()

model.produtos = pyo.Set(initialize=produtos)
model.requisitos = pyo.Set(initialize=requisitos)
model.x = pyo.Var(model.produtos, bounds=(0,None), domain=pyo.NonNegativeIntegers)

def obj(model):
    return sum(
        model.x[p] * informacoes[p]['lucro'] for p in model.produtos
    )
model.obj = pyo.Objective(rule=obj, sense=pyo.maximize)

def restricoes(model, r):
    return sum(
        model.x[p] * informacoes[p][r] for p in model.produtos
    ) <= valores_req[r]
model.req = pyo.Constraint(model.requisitos, rule=restricoes)

In [10]:
model.pprint()

2 Set Declarations
    produtos : Size=1, Index=None, Ordered=Insertion
        Key  : Dimen : Domain : Size : Members
        None :     1 :    Any :    2 : {'portas', 'janelas'}
    requisitos : Size=1, Index=None, Ordered=Insertion
        Key  : Dimen : Domain : Size : Members
        None :     1 :    Any :    3 : {'corte', 'jateamento', 'acabamento'}

1 Var Declarations
    x : Size=2, Index=produtos
        Key     : Lower : Value : Upper : Fixed : Stale : Domain
        janelas :     0 :  None :  None : False :  True : NonNegativeIntegers
         portas :     0 :  None :  None : False :  True : NonNegativeIntegers

1 Objective Declarations
    obj : Size=1, Index=None, Active=True
        Key  : Active : Sense    : Expression
        None :   True : maximize : 500*x[portas] + 400*x[janelas]

1 Constraint Declarations
    req : Size=3, Index=requisitos, Active=True
        Key        : Lower : Body                         : Upper  : Active
        acabamento :  -Inf : 30*x[port

In [11]:
opt = SolverFactory('cplex')
res = opt.solve(model, tee=True)


Welcome to IBM(R) ILOG(R) CPLEX(R) Interactive Optimizer 22.1.1.0
  with Simplex, Mixed Integer & Barrier Optimizers
5725-A06 5725-A29 5724-Y48 5724-Y49 5724-Y54 5724-Y55 5655-Y21
Copyright IBM Corp. 1988, 2022.  All Rights Reserved.

Type 'help' for a list of available commands.
Type 'help' followed by a command name for more
information on commands.

CPLEX> Logfile 'cplex.log' closed.
Logfile 'C:\Users\DECIV\AppData\Local\Temp\tmp6r3g7s0g.cplex.log' open.
CPLEX> Problem 'C:\Users\DECIV\AppData\Local\Temp\tmpkc1_vn93.pyomo.lp' read.
Read time = 0.00 sec. (0.00 ticks)
CPLEX> Problem name         : C:\Users\DECIV\AppData\Local\Temp\tmpkc1_vn93.pyomo.lp
Objective sense      : Maximize
Variables            :       2  [General Integer: 2]
Objective nonzeros   :       2
Linear constraints   :       3  [Less: 3]
  Nonzeros           :       6
  RHS nonzeros       :       3

Variables            : Min LB: 0.000000         Max UB: all infinite   
Objective nonzeros   : Min   : 400.0000       

In [13]:
for p in model.x:
    print(f"Quantidade de {p} = {pyo.value(model.x[p]):.2f}")
print(f"Lucro total de R$ {model.obj():.2f}")

Quantidade de portas = 20.00
Quantidade de janelas = 40.00
Lucro total de R$ 26000.00
